# ngen-verf Tutorial

### Introduction to ngen-verf

ngen-verf is a command-line tool designed for downloading and verifying NWM streamflow forecasts, as well as comparing statistics from different forecast datasets or time periods. 



Five steps to verification using ngen-verf:
- Fetch NWM streamflow forecast data (from Google Cloud using TEEHR library)
- Fetch streamflow observations (USGS)
- Pair observations with forecasts
- Compute evaluation metrics
- Generate visualization (spatial maps, boxplots, histograms)

In [2]:
# load libraries
from pathlib import Path
import pandas as pd
from funcs_ngen_verf_demo import display_images, display_config, update_yaml_field

### Examine different sections of config file

There are 7 different sections in the config file: general, file_paths, NWM_forecast, flow_observation, pair_data, metrics, plots

In [ ]:
config_file = "/home/yuqiong.liu/work/Gitlab/ngen-verf/sample_files/config.yaml"
display_config(config_file, 'general')

In [ ]:
display_config(config_file, 'file_paths')

In [ ]:
display_config(config_file, 'nwm_forecast')

In [ ]:
display_config(config_file, 'flow_observation')

In [ ]:
display_config(config_file, 'pair_data')

In [ ]:
display_config(config_file, 'metrics')

In [ ]:
display_config(config_file, 'plots')

### Use Case 1: download and pair data
Downloading and processing NWM forecasts can be very resource intensive and time consuming. 

For demonstration, we will:
- download and process short-range forecasts for just a couple of days 
- download USGS observations for the corresponding days
- create the paired data
- skip metrics calcualtion and visualization given the short time period

##### Adjust configurations

In [23]:
# make a copy of the config file for the current use case
!cp /home/yuqiong.liu/work/Gitlab/run/ngen_verf_config.yaml config_case1.yaml
config_file = "config_case1.yaml"

In [24]:
# update config file to download data to a new directory and to skip the last two steps (compute and plot metrics) 
update_yaml_field(config_file, ['general', 'location_set_name'], "new_group")
update_yaml_field(config_file, ['general', 'forecast_end_date'], ['2024-09-01', '2024-10-01'])
update_yaml_field(config_file, ['general', 'steps', 'compute_metrics'], False)
update_yaml_field(config_file, ['general', 'steps', 'plot_metrics'], False)

In [ ]:
# check the updated config file
display_config(config_file, 'general')

##### Run verification

In [ ]:
# run the verification program
!cp /home/yuqiong.liu/work/Gitlab/ngen-verf/verification.py .
!python verification.py config_case1.yaml 

##### Check outputs

In [ ]:
# check outputs in the new directory 'new_group
!tree -L 2 /home/yuqiong.liu/work/data/ngen-verf/new_group/

### Use Case 2: compute and visualize statistics

Compute a few metrics (KGE, NSE, NNSE, CORR) using previously processed data and generated visualization plots

In [13]:
# check previously downloaded and paired datasets
!tree -L 2 /home/yuqiong.liu/work/data/ngen-verf/calib_basin_group1/

/home/yuqiong.liu/work/data/ngen-verf/calib_basin_group1/
|-- joined
|   |-- v3_oct.joined.parquet
|   |-- v3_oct.joined_new.parquet
|   |-- v3_sep.joined.parquet
|   `-- v3_sep.joined_new.parquet
|-- metrics
|   |-- v3_oct.metrics.parquet
|   `-- v3_sep.metrics.parquet
|-- nwm30
|   |-- timeseries
|   `-- zarr
|-- plots
|   |-- boxplots
|   |-- histograms
|   `-- maps
|-- usgs
|   |-- 2022-10-01_2022-10-31.parquet
|   |-- 2022-11-01_2022-11-01.parquet
|   |-- 2024-09-01_2024-09-30.parquet
|   |-- 2024-10-01_2024-10-31.parquet
|   |-- 2024-11-01_2024-11-12.parquet
|   `-- 2024-11-01_2024-11-19.parquet
|-- v3_oct
|   `-- fcst
`-- v3_sep
    `-- fcst

15 directories, 12 files


In [14]:
# make a copy of config file for Use Case 2
!cp /home/yuqiong.liu/work/Gitlab/run/ngen_verf_config.yaml config_case2.yaml
config_file = 'config_case2.yaml'

In [15]:
# update the config file to skip the first three steps and only run the last two steps: compute and plot metrics
update_yaml_field(config_file, ['general', 'steps', 'fetch_fcst_data'], False)
update_yaml_field(config_file, ['general', 'steps', 'fetch_obs_data'], False)
update_yaml_field(config_file, ['general', 'steps', 'pair_data'], False)

In [16]:
# run verification program
!python verification.py config_case2.yaml 

INFO:__main__:Config file to use: config_case2.yaml
INFO:ngen.verf.calc_metrics:  Calculating metrics using ngen.eval library
INFO:ngen.verf.calc_metrics:  Calculating metrics using ngen.eval library
INFO:ngen.verf.calc_metrics:  Metrics for dataset v3_sep are save at /home/yuqiong.liu/work/data/ngen-verf/calib_basin_group1/metrics/v3_sep.metrics.parquet
INFO:ngen.verf.calc_metrics:  Calculating metrics using ngen.eval library
INFO:ngen.verf.calc_metrics:  Calculating metrics using ngen.eval library
INFO:ngen.verf.calc_metrics:  Metrics for dataset v3_oct are save at /home/yuqiong.liu/work/data/ngen-verf/calib_basin_group1/metrics/v3_oct.metrics.parquet
INFO:__main__:Execution time for compute_metrics: 24.73240876197815 seconds
INFO:ngen.verf.create_plots:Spatial maps created at: /home/yuqiong.liu/work/data/ngen-verf/calib_basin_group1/plots/maps
INFO:ngen.verf.create_plots:Histograms created at: /home/yuqiong.liu/work/data/ngen-verf/calib_basin_group1/plots/histograms
INFO:ngen.verf.c

In [17]:
# check the output directory (with newly added metrics and plots)
!tree -L 2 /home/yuqiong.liu/work/data/ngen-verf/calib_basin_group1

/home/yuqiong.liu/work/data/ngen-verf/calib_basin_group1
|-- joined
|   |-- v3_oct.joined.parquet
|   |-- v3_oct.joined_new.parquet
|   |-- v3_sep.joined.parquet
|   `-- v3_sep.joined_new.parquet
|-- metrics
|   |-- v3_oct.metrics.parquet
|   `-- v3_sep.metrics.parquet
|-- nwm30
|   |-- timeseries
|   `-- zarr
|-- plots
|   |-- boxplots
|   |-- histograms
|   `-- maps
|-- usgs
|   |-- 2022-10-01_2022-10-31.parquet
|   |-- 2022-11-01_2022-11-01.parquet
|   |-- 2024-09-01_2024-09-30.parquet
|   |-- 2024-10-01_2024-10-31.parquet
|   |-- 2024-11-01_2024-11-12.parquet
|   `-- 2024-11-01_2024-11-19.parquet
|-- v3_oct
|   `-- fcst
`-- v3_sep
    `-- fcst

15 directories, 12 files


In [18]:
# check the plots directory
!tree -L 2 /home/yuqiong.liu/work/data/ngen-verf/calib_basin_group1/plots

/home/yuqiong.liu/work/data/ngen-verf/calib_basin_group1/plots
|-- boxplots
|   |-- boxplot_CORR.png
|   |-- boxplot_CSI.png
|   |-- boxplot_FAR.png
|   |-- boxplot_KGE.png
|   |-- boxplot_NNSE.png
|   |-- boxplot_NSE.png
|   |-- boxplot_PBIAS.png
|   `-- boxplot_POD.png
|-- histograms
|   |-- hist_CORR_h1-5.png
|   |-- hist_CORR_h1.png
|   |-- hist_CORR_h10.png
|   |-- hist_CORR_h15.png
|   |-- hist_CORR_h18.png
|   |-- hist_CORR_h3.png
|   |-- hist_CORR_h5.png
|   |-- hist_CSI_h1-5.png
|   |-- hist_CSI_h1.png
|   |-- hist_CSI_h10.png
|   |-- hist_CSI_h11-15.png
|   |-- hist_CSI_h15.png
|   |-- hist_CSI_h18.png
|   |-- hist_CSI_h3.png
|   |-- hist_CSI_h5.png
|   |-- hist_CSI_h6-10.png
|   |-- hist_FAR_h1-5.png
|   |-- hist_FAR_h1.png
|   |-- hist_FAR_h10.png
|   |-- hist_FAR_h11-15.png
|   |-- hist_FAR_h15.png
|   |-- hist_FAR_h18.png
|   |-- hist_FAR_h3.png
|   |-- hist_FAR_h5.png
|   |-- hist_FAR_h6-10.png
|   |-- hist_KGE_h1-5.png
|   |-- hist_KGE_h1.png
|   |-- hist_KGE_h10.png
| 

In [19]:
# display boxplot plots for KGE and NSE
plot_dir = "/home/yuqiong.liu/work/data/ngen-verf/calib_basin_group1/plots"
display_images(Path(plot_dir), 'boxplot', ['KGE', 'NSE'])

In [20]:
# display hisgrams for two lead times: h1, h10
display_images(Path(plot_dir), 'histogram', ['KGE'], ['h1', 'h10'])

In [21]:
# display spatial maps for h1 NSE for the two datasets
display_images(Path(plot_dir), 'map', ['NSE'], ['h1'], ['v3_sep','v3_oct'])

### Use Case 3: Compute additional metrics and lead times
Compute a few additional metrics (PBIAS, POD, CSI, FAR) and lead times (6-10, 11-15)

In [ ]:
# make a copy of config file for Use Case 3
!cp config_case2.yaml config_case3.yaml
config_file = 'config_case3.yaml'

# check settings for current metrics and lead times
display_config(config_file, 'metrics')

In [5]:
# update lead times and metrics to compute and plot
update_yaml_field(config_file, ['metrics', 'metric_subset'], ['PBIAS','POD','FAR','CSI'])
update_yaml_field(config_file, ['metrics', 'lead_times'], ['all', '1-5', '6-10', '11-15'])
update_yaml_field(config_file, ['plots', 'boxplot','metric_subset'], ['PBIAS','POD','FAR','CSI'])
update_yaml_field(config_file, ['plots', 'boxplot','lead_times'], [1, 3, 5, 10, 15, 18, '1-5', '6-10', '11-15'])
update_yaml_field(config_file, ['plots', 'histogram','metric_subset'], ['PBIAS','POD','FAR','CSI'])
update_yaml_field(config_file, ['plots', 'histogram','lead_times'], [1, 3, 5, 10, 15, 18,'1-5', '6-10', '11-15'])
update_yaml_field(config_file, ['plots', 'spatial_map','plot'], False)

In [ ]:
# run verification program
!python verification.py config_case3.yaml 

In [ ]:
# display boxplot plots for POD and FAR
plot_dir = "/home/yuqiong.liu/work/data/ngen-verf/calib_basin_group1/plots"
display_images(Path(plot_dir), 'boxplot', ['POD', 'FAR'])

In [ ]:
# display histogram for PBIAS
display_images(Path(plot_dir), 'histogram', ['PBIAS'],['h1','h10'])

In [ ]:
# check plot settings
display_config(config_file, 'plots')

In [10]:
# check metric values to derive reasonable bin edges for histograms
f1 = "/home/yuqiong.liu/work/data/ngen-verf/calib_basin_group1/metrics/v3_oct.metrics.parquet"
display(pd.read_parquet(f1).describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

,PBIAS,POD,FAR,CSI
count,1827.000000,1791.000000,1632.000000,1995.000000
mean,117.168140,0.618736,0.605974,0.228366
std,667.713364,0.426593,0.365954,0.266499
min,-102.333426,0.000000,0.000000,0.000000
5%,-75.826486,0.000000,0.000000,0.000000
25%,-15.817245,0.041667,0.277778,0.000000
50%,7.601192,0.866667,0.768183,0.100806
75%,68.058819,1.000000,0.908284,0.388835
95%,252.211032,1.000000,1.000000,0.815389
max,13761.480713,1.000000,1.000000,0.986842


In [ ]:
# add bin edges for PBIAS in config file
update_yaml_field(config_file, ['plots', 'histogram','binning','PBIAS'], ['-inf', -100, -20, 20, 60, 100, 200,  'inf'])
update_yaml_field(config_file, ['general', 'steps', 'compute_metrics'], False)

# run verification program
!python verification.py config_case3.yaml 

In [ ]:
# check histograms of PBIAS again
display_images(Path(plot_dir), 'histogram', ['PBIAS'], ['h1','h10'])